### 粗粒度分子构建原始


In [ ]:
from builder.aromatic_planner import phase1_run

csv_path = "inverse_result/layer3_nodes.csv"
dist = phase1_run(
    csv_path,
    out_dir="viz_phase1",
    tag="phase1",
    visualize_raw=True,        
    raw_out_dir="viz_phase1_raw"  
)
print("Phase1 分布统计:", dist)

In [ ]:
### Phase2: 刚性连接构建（仅显示刚性连接）
from builder.aromatic_planner import phase2_run 
from builder.aromatic_planner import load_su_counts_from_csv

counts = load_su_counts_from_csv("inverse_result/layer3_nodes.csv")
print("结构单元统计:", counts)
print("可用于刚性连接的 10 号:", counts.get(10, 0))
print()

summary, conn, graph_dict, action_plan = phase2_run(
    "inverse_result/layer3_nodes.csv",
    seed=780,
    out_dir="viz_phase2",
    tag="phase2",
    visualize_mode="rigid"   
)

print("前5个动作计划:", action_plan[:5]) 

In [ ]:
from RL.aromatic_planner import phase2_run, load_su_counts_from_csv

counts = load_su_counts_from_csv("inverse_result/layer3_nodes.csv")
print("结构单元统计:", counts)
print("可用于柔性连接的 11 号:", counts.get(11, 0))
print("可用于柔性连接的 23 号:", counts.get(23, 0))
print()

summary_flex, conn_flex, graph_dict_flex, action_plan_flex = phase2_run(
    "inverse_result/layer3_nodes.csv",
    seed=1802,
    out_dir="viz_phase2_rigid",
    tag="phase2",
    visualize_mode="flex",
    flex_out_dir="viz_phase2_flex",
    flex_tag="phase2_flex",
    flex_show_side_chains=True,
    flex_show_rigid_edges=True,
)

print("柔性阶段前5个动作计划:", action_plan_flex[:5])

In [ ]:
from RL.aromatic_planner import phase2_run, load_su_counts_from_csv

summary, conn, graph_dict, action_plan = phase2_run(
    "inverse_result/layer3_nodes.csv", 
    seed=1802,
    enable_substitution=True,
    visualize_mode="flex",
    flex_out_dir="viz_simple_oxygen",
)

### MCTS验证

In [ ]:
from RL.test_mcts import test_flex_mcts

results = test_flex_mcts(
    csv_path="inverse_result/layer3_nodes.csv",
    n_rigid_iters=100,
    n_flex_iters=100,
    n_side_iters=60,
    n_2425_iters=50,
    n_secondary_iters=50,
    n_final_subst_iters=30,
    output_dir="test_flex_results-complete",
    enable_side=True,
    enable_2425=True,
    enable_secondary_flex=True,
    enable_final_substitution=True,
)

### 强化学习

In [ ]:
from RL.mcts_search import HierarchicalBeamMCTS

mcts = HierarchicalBeamMCTS(
    csv_path="test_results/1/final_outputs/final_nodes.csv",
    beam_width=5,
    stage_iterations={
        'rigid': 20, 'flex': 20, 'side': 20,
        'branch_sub': 20, 'secondary_flex': 20, 'substitution': 20
    },
    nmr_model_ckpt="checkpoints_g2s/g2s_best_model.pt",
    target_spectrum="./standardized_nmr.csv",  
    enable_nmr_eval=True,
    elements_str="C=460,H=400,O=60,N=2,S=2,X=0", 
)

mcts.BACKTRACK_CONFIG['nmr_threshold'] = 0.8
mcts.BACKTRACK_CONFIG['stage_lookahead_threshold'] = 0.86
mcts.BACKTRACK_CONFIG['max_backtracks'] = 8
mcts.BACKTRACK_CONFIG['per_stage_check'] = False

result = mcts.run(output_dir="RL_file/test_results_17")

### 粗粒度分子的转化

In [ ]:
# 生成简化 JSON / TXT / 粗粒度2D图
!python3 cg2mol/cg_simplify.py input_cg.json -t -p

In [ ]:
# 简化 CG → 全原子 JSON：
!python3 cg2mol/cg_to_allatom.py \
  RL_file/test_results_12/coarse_grained_molecules/cg_molecule_1_9b205314_simplified.json \
  -v

In [ ]:
# 导出2D全原子分子
!python3 cg2mol/allatom_to_rdkit.py \
  RL_file/test_results_12/coarse_grained_molecules/cg_molecule_1_9b205314_allatom.json \
  -c 2d -f sdf pdb -v

In [ ]:
# 导出3D全原子分子
!python3 cg2mol/allatom_to_rdkit.py \
  RL_file/test_results_12/coarse_grained_molecules/cg_molecule_1_9b205314_allatom.json \
  -c 3d -f sdf pdb -v

### 代码重构

In [ ]:
# 初始化
import sys
sys.path.insert(0, 'RL_MTCS')
from RL_init import initialize
from visualization import save_all_clusters, save_overview

result = initialize('test_results/1-7/final_outputs/final_nodes.csv', 
                    spectrum_csv='./standardized_nmr.csv') 

save_all_clusters(result['clusters'], 'output_dir1/')

In [ ]:
# 刚性连接
import sys
sys.path.insert(0, 'RL_MTCS')
from RL_MTCS import MCTSConfig, MCTSRunner

config = MCTSConfig(
    iterations=1,     
    beam_width=1,      
    max_cluster_size=4,  
    seed=991
)

runner = MCTSRunner(
    nodes_csv='test_results/1-7/final_outputs/final_nodes.csv',
    config=config
)

result = runner.run_rigid_stage(output_dir='output_rigid3/')

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg' 

from RL_MTCS.RL_mcts import MCTSRunner, MCTSConfig

config = MCTSConfig(
    nmr_model_ckpt="checkpoints_g2s/g2s_best_model.pt",
    spectrum_csv="/mnt/e/NN/GA/MAS/standardized_nmr.csv",
    iterations=2,     
    beam_width=2,      
    max_cluster_size=4,  
    seed=919,
    flex_n_seeds=1,       
    flex_iterations=7,   
    flex_max_steps=50,    
    flex_debug=True,
    side_n_seeds=1, 
    side_beam_width=1,
    nmr_weight=20.0
)
runner = MCTSRunner('test_results/MAS-3/final_outputs/final_nodes.csv', None, config)
final_result = runner.run_all_stages(output_dir='test_RL/MAS-4')

best_side = final_result.get('best_candidate')
if best_side:
    print(f"最终 Score: {best_side.score:.4f}")
    side_info = best_side.info.get('side_result', {})
    print(f"侧链放置完成数: {side_info.get('sides_placed', 0)} / {side_info.get('sides_total', 0)}")

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'

from RL_MTCS.RL_mcts import MCTSRunner, MCTSConfig

config = MCTSConfig(
    nmr_model_ckpt="checkpoints_g2s/g2s_best_model.pt",
    target_spectrum="/mnt/e/NN/GA/MAS/standardized_nmr.csv",

    iterations=10,
    beam_width=1,
    max_cluster_size=4,
    seed=1000,

    flex_iterations=6,
    flex_max_steps=40,
    flex_beam_width=1,

    branch_iterations=25,
    side_iterations=30,
    subst_iterations=50,

    branch_candidate_k=8,
    side_candidate_k=8,
    subst_candidate_k=6,
    stage_action_branching=2,
    side_beam_width=2,
    subst_n_variants=3,

    nmr_weight=20.0,
)

runner = MCTSRunner(
    'test_results/MAS-1/final_outputs/final_nodes.csv',
    None,
    config
)

final_result = runner.run_all_stages(output_dir='test_RL/MAS-1')

best_final = final_result.get('best_candidate')
if best_final:
    print(f"最终 Score: {best_final.score:.4f}")

    flex_info = best_final.info.get('flex_result', {})
    branch_info = best_final.info.get('branch_result', {})
    side_info = best_final.info.get('side_result', {})
    subst_info = best_final.info.get('subst_result', {})

    print(f"柔性连接完成: {flex_info.get('bridges_done', 0)} / {flex_info.get('bridges_total', 0)}")
    print(f"分支放置完成: {branch_info.get('branches_placed', 0)} / {branch_info.get('branches_total', 0)}")
    print(f"侧链放置完成: {side_info.get('sides_placed', 0)} / {side_info.get('sides_total', 0)}")
    print(f"取代剩余数: {subst_info.get('remaining_total', 0)}")
